# 面试问题：S-LoRA 怎样共享基础模型并并发服务多租户、多版本 Adapter？

        ## 可直接复述的回答主线

        1. LoRA 把每个任务的权重增量写成低秩矩阵 A@B；S-LoRA 只保留一份基础权重，并在请求路径动态应用对应增量。
2. 朴素方案为每个 Adapter 物化一份 W+AB，会重复保存巨大的基础权重；共享方案的常驻内存是一个 W 加各低秩矩阵。
3. 请求键必须包含 tenant、adapter_id 和 version，只按名称索引会发生跨租户串权重或旧版本覆盖。
4. 运行时用容量受限的 Adapter Pool 加载低秩矩阵，记录 hit、load、evict，并用 LRU 演示换入换出。
5. 批处理先共享计算 X@W，再按 rank/version 分组计算 (X@A)@B；结果应与物化全权重逐请求计算一致。
6. 生产实现还需要 GPU 连续内存池、融合 kernel、请求公平性、版本 pin、鉴权、热度预取和尾延迟监控。

        后续实验会在同一批输入上依次展示朴素基线、手写核心机制、中间过程、失败修正和生产边界。

## 1. 真实案例与输入预览

案例包含三个租户、六个 Adapter 和八条混合请求。`support-tone` 同时存在于 tenant-a 的 v1/v2 和 tenant-b 的 v1，用来真实复现仅按名称或忽略版本索引时的串权重；池容量只有三个 Adapter，因此还会出现命中与 LRU 淘汰。

In [1]:
import numpy as np  # 使用基础数组手写低秩增量、批处理和 LRU Adapter Pool。
rng = np.random.default_rng(30)  # 固定基础权重、Adapter 和请求输入以保证结果可复现。
input_dim = 16  # 设置教学线性层输入维度以体现低秩节省。
output_dim = 12  # 设置教学线性层输出维度。
base_weight = rng.normal(0.0, 0.18, size=(input_dim, output_dim)).astype(np.float64)  # 创建所有租户共享的一份基础权重。
adapter_specs = [{"tenant": "tenant-a", "adapter_id": "support-tone", "version": 1, "rank": 2}, {"tenant": "tenant-a", "adapter_id": "support-tone", "version": 2, "rank": 2}, {"tenant": "tenant-a", "adapter_id": "legal-brief", "version": 2, "rank": 1}, {"tenant": "tenant-c", "adapter_id": "retail-copy", "version": 3, "rank": 2}, {"tenant": "tenant-b", "adapter_id": "sql-helper", "version": 1, "rank": 1}, {"tenant": "tenant-b", "adapter_id": "support-tone", "version": 1, "rank": 2}]  # 定义含同名跨租户和同租户双版本的六个 Adapter。
adapters = []  # 保存每个 Adapter 的低秩 A、B 参数。
for index, spec in enumerate(adapter_specs):  # 按规格生成确定性的低秩增量。
    matrix_a = rng.normal(0.0, 0.09 + index * 0.005, size=(input_dim, spec["rank"])).astype(np.float64)  # 创建输入到低秩空间的 A。
    matrix_b = rng.normal(0.0, 0.11 + index * 0.004, size=(spec["rank"], output_dim)).astype(np.float64)  # 创建低秩空间到输出的 B。
    key = (spec["tenant"], spec["adapter_id"], spec["version"])  # 构造包含租户和版本的完整键。
    adapters.append({**spec, "key": key, "A": matrix_a, "B": matrix_b})  # 保存元数据和低秩矩阵。
adapter_registry = {adapter["key"]: adapter for adapter in adapters}  # 建立无冲突的版本化 Adapter 注册表。
request_specs = [{"id": "slora-01", "key": ("tenant-a", "support-tone", 1)}, {"id": "slora-02", "key": ("tenant-a", "support-tone", 2)}, {"id": "slora-03", "key": ("tenant-a", "support-tone", 1)}, {"id": "slora-04", "key": ("tenant-a", "legal-brief", 2)}, {"id": "slora-05", "key": ("tenant-c", "retail-copy", 3)}, {"id": "slora-06", "key": ("tenant-b", "sql-helper", 1)}, {"id": "slora-07", "key": ("tenant-b", "support-tone", 1)}, {"id": "slora-08", "key": ("tenant-a", "support-tone", 2)}]  # 定义会产生池命中和淘汰的八条请求顺序。
requests = []  # 保存请求向量和目标 Adapter 键。
for index, spec in enumerate(request_specs):  # 为每条请求生成不同业务输入向量。
    input_vector = rng.normal(loc=index * 0.01, scale=0.35, size=(input_dim,)).astype(np.float64)  # 创建当前请求的十六维隐藏状态。
    requests.append({**spec, "input": input_vector})  # 保存请求 ID、版本化键和输入。
print("教学实验输入：六个 Adapter 与八条请求")  # 标记下方为多租户 Serving 案例。
print("Adapter key                                      rank   A shape    B shape")  # 输出 Adapter 清单表头。
for adapter in adapters:  # 逐个展示租户、名称、版本和低秩形状。
    print(f"{str(adapter['key']):<48} {adapter['rank']:>4}   {str(adapter['A'].shape):<10} {adapter['B'].shape}")  # 输出当前 Adapter 参数结构。
print("请求顺序：", [(request["id"], request["key"], np.round(request["input"][:3], 3).tolist()) for request in requests])  # 展示池访问顺序和输入前三维。

教学实验输入：六个 Adapter 与八条请求
Adapter key                                      rank   A shape    B shape
('tenant-a', 'support-tone', 1)                     2   (16, 2)    (2, 12)
('tenant-a', 'support-tone', 2)                     2   (16, 2)    (2, 12)
('tenant-a', 'legal-brief', 2)                      1   (16, 1)    (1, 12)
('tenant-c', 'retail-copy', 3)                      2   (16, 2)    (2, 12)
('tenant-b', 'sql-helper', 1)                       1   (16, 1)    (1, 12)
('tenant-b', 'support-tone', 1)                     2   (16, 2)    (2, 12)
请求顺序： [('slora-01', ('tenant-a', 'support-tone', 1), [-0.238, 0.166, 0.563]), ('slora-02', ('tenant-a', 'support-tone', 2), [0.625, 0.25, 0.263]), ('slora-03', ('tenant-a', 'support-tone', 1), [0.221, 0.074, 0.401]), ('slora-04', ('tenant-a', 'legal-brief', 2), [-0.154, -0.186, -0.852]), ('slora-05', ('tenant-c', 'retail-copy', 3), [-0.231, 0.242, 0.283]), ('slora-06', ('tenant-b', 'sql-helper', 1), [-0.47, 0.311, -0.382]), ('slora-07', ('tenant-b

## 2. Baseline / 基线：为每个 Adapter 物化完整权重

基线先计算 `W_adapter = W_base + A@B`，每个 Adapter 都复制 16×12 的基础层。八条请求随后各自做一次完整矩阵乘法。

In [2]:
materialized_weights = {}  # 保存六个 Adapter 各自的完整权重副本。
for adapter in adapters:  # 逐 Adapter 物化 W 加低秩增量。
    materialized_weights[adapter["key"]] = base_weight + adapter["A"] @ adapter["B"]  # 计算并保存完整租户权重。
baseline_outputs = {}  # 保存八条请求的基线输出。
for request in requests:  # 逐请求选择完整权重执行前向。
    full_weight = materialized_weights[request["key"]]  # 按完整版本化键读取物化权重。
    baseline_outputs[request["id"]] = request["input"] @ full_weight  # 计算当前请求十二维输出。
baseline_resident_bytes = sum(weight.nbytes for weight in materialized_weights.values())  # 统计六份完整权重的常驻字节数。
print("Baseline 物化输出预览")  # 标记下表展示真实数值而非只有内存公式。
print("请求       Adapter key                                      output前3维")  # 输出基线结果表头。
for request in requests:  # 逐条展示使用的版本和输出片段。
    print(f"{request['id']:<10} {str(request['key']):<48} {np.round(baseline_outputs[request['id']][:3], 6).tolist()}")  # 输出当前请求基线结果。
print(f"六份完整权重常驻内存={baseline_resident_bytes} bytes")  # 展示重复基础权重带来的内存开销。

Baseline 物化输出预览
请求       Adapter key                                      output前3维
slora-01   ('tenant-a', 'support-tone', 1)                  [-0.181879, -0.169944, 0.660235]
slora-02   ('tenant-a', 'support-tone', 2)                  [0.450442, -0.054941, 0.274626]
slora-03   ('tenant-a', 'support-tone', 1)                  [-0.382166, -0.32591, -0.178567]
slora-04   ('tenant-a', 'legal-brief', 2)                   [-0.033927, 0.300206, 0.091772]
slora-05   ('tenant-c', 'retail-copy', 3)                   [-0.059503, -0.282149, 0.331016]
slora-06   ('tenant-b', 'sql-helper', 1)                    [-0.201497, -0.202598, 0.261097]
slora-07   ('tenant-b', 'support-tone', 1)                  [-0.167021, -0.331977, -0.496752]
slora-08   ('tenant-a', 'support-tone', 2)                  [-0.025417, -0.219938, 0.079563]
六份完整权重常驻内存=9216 bytes


## 3. 底层实现：共享 base、低秩 delta、版本化 LRU Pool 与分组

下面显式实现 `X@W + (X@A)@B`。Pool 只保存低秩矩阵，按完整键查找；每次访问都会输出 hit/load/evict。分组表显示运行时怎样把相同 rank/version 的请求放到同一执行桶。

In [3]:
class AdapterPool:  # 手写容量受限且按版本化键索引的 LRU Adapter Pool。
    def __init__(self, registry, capacity):  # 初始化后端注册表和最大驻留数量。
        self.registry = registry  # 保存完整 Adapter 参数来源。
        self.capacity = capacity  # 保存同时驻留的最大 Adapter 数。
        self.entries = {}  # 保存当前驻留 Adapter 及最近访问时钟。
        self.clock = 0  # 使用确定性逻辑时钟实现 LRU。
        self.events = []  # 保存每次 hit、load 和 evict 事件。
    def get(self, key):  # 获取指定租户、名称和版本的 Adapter。
        self.clock += 1  # 每个请求推进一次逻辑时钟。
        if key in self.entries:  # 检查 Adapter 是否已经驻留。
            self.entries[key]["last_used"] = self.clock  # 更新命中条目的最近使用时刻。
            event = {"action": "hit", "key": key, "evicted": None, "resident": list(self.entries)}  # 构造命中事件。
            self.events.append(event)  # 保存命中账本。
            return self.entries[key]["adapter"], event  # 返回驻留参数和事件。
        evicted = None  # 默认本次加载不淘汰条目。
        if len(self.entries) >= self.capacity:  # 检查低秩参数池是否已满。
            evicted = min(self.entries, key=lambda resident_key: self.entries[resident_key]["last_used"])  # 找到最久未使用的版本化键。
            del self.entries[evicted]  # 从驻留池移除 LRU Adapter。
        adapter = self.registry[key]  # 从后端注册表加载正确租户和版本的参数。
        self.entries[key] = {"adapter": adapter, "last_used": self.clock}  # 把新 Adapter 放入池并记录时钟。
        event = {"action": "load", "key": key, "evicted": evicted, "resident": list(self.entries)}  # 构造加载和可能的淘汰事件。
        self.events.append(event)  # 保存换入换出账本。
        return adapter, event  # 返回新加载参数和事件。
input_matrix = np.stack([request["input"] for request in requests])  # 把八条输入堆叠为共享基础批次。
shared_base_outputs = input_matrix @ base_weight  # 一次批矩阵乘计算所有请求的基础输出。
groups = {}  # 按 rank 和 version 保存调度分组。
for index, request in enumerate(requests):  # 遍历请求以构造兼容的低秩执行桶。
    adapter = adapter_registry[request["key"]]  # 读取当前请求 Adapter 元数据。
    group_key = (adapter["rank"], adapter["version"])  # 用 rank 和版本构造教学分组键。
    groups.setdefault(group_key, []).append(index)  # 把请求下标加入对应执行桶。
pool = AdapterPool(adapter_registry, capacity=3)  # 创建只能驻留三个低秩 Adapter 的池。
slora_outputs = {}  # 保存共享基础加动态增量的最终输出。
request_events = {}  # 保存每条请求的 Pool 事件。
for index, request in enumerate(requests):  # 按到达顺序服务八条请求。
    adapter, event = pool.get(request["key"])  # 从版本化 Pool 获取低秩参数。
    low_rank_state = request["input"] @ adapter["A"]  # 先把输入投影到 rank 维空间。
    delta_output = low_rank_state @ adapter["B"]  # 再把低秩状态投影到输出维度。
    slora_outputs[request["id"]] = shared_base_outputs[index] + delta_output  # 将共享基础输出与当前 Adapter 增量相加。
    request_events[request["id"]] = event  # 保存命中、加载或淘汰证据。
print("rank/version 调度分组：", {str(key): [requests[index]["id"] for index in indices] for key, indices in groups.items()})  # 展示八条请求的分组结果。
print("LRU Pool 事件")  # 标记下表展示真实换入换出过程。
for request in requests:  # 按请求顺序打印对应 Pool 事件。
    event = request_events[request["id"]]  # 读取当前请求事件。
    print(f"{request['id']} action={event['action']:<4} key={event['key']} evicted={event['evicted']} resident={event['resident']}")  # 输出命中、淘汰和驻留集合。

rank/version 调度分组： {'(2, 1)': ['slora-01', 'slora-03', 'slora-07'], '(2, 2)': ['slora-02', 'slora-08'], '(1, 2)': ['slora-04'], '(2, 3)': ['slora-05'], '(1, 1)': ['slora-06']}
LRU Pool 事件
slora-01 action=load key=('tenant-a', 'support-tone', 1) evicted=None resident=[('tenant-a', 'support-tone', 1)]
slora-02 action=load key=('tenant-a', 'support-tone', 2) evicted=None resident=[('tenant-a', 'support-tone', 1), ('tenant-a', 'support-tone', 2)]
slora-03 action=hit  key=('tenant-a', 'support-tone', 1) evicted=None resident=[('tenant-a', 'support-tone', 1), ('tenant-a', 'support-tone', 2)]
slora-04 action=load key=('tenant-a', 'legal-brief', 2) evicted=None resident=[('tenant-a', 'support-tone', 1), ('tenant-a', 'support-tone', 2), ('tenant-a', 'legal-brief', 2)]
slora-05 action=load key=('tenant-c', 'retail-copy', 3) evicted=('tenant-a', 'support-tone', 2) resident=[('tenant-a', 'support-tone', 1), ('tenant-a', 'legal-brief', 2), ('tenant-c', 'retail-copy', 3)]
slora-06 action=load key=('

## 4. 逐请求结果与结果解读

逐请求比较物化权重和共享低秩路径的最大绝对差。目录内存按 NumPy FP64 实际字节统计；活动 Pool 内存另按最后三个驻留 Adapter 计算。

In [4]:
result_rows = []  # 保存八条请求的数值一致性和 Pool 行为。
for request in requests:  # 逐请求比较两条前向路径。
    baseline_output = baseline_outputs[request["id"]]  # 读取物化权重输出。
    slora_output = slora_outputs[request["id"]]  # 读取共享基础加低秩增量输出。
    maximum_difference = float(np.max(np.abs(baseline_output - slora_output)))  # 计算十二维输出最大绝对差。
    result_rows.append({"id": request["id"], "key": request["key"], "difference": maximum_difference, "action": request_events[request["id"]]["action"], "evicted": request_events[request["id"]]["evicted"]})  # 保存逐请求可解释结果。
adapter_parameter_bytes = sum(adapter["A"].nbytes + adapter["B"].nbytes for adapter in adapters)  # 统计目录中全部低秩参数字节数。
slora_catalog_bytes = base_weight.nbytes + adapter_parameter_bytes  # 统计一份基础权重加六个低秩 Adapter 的目录内存。
active_adapter_bytes = sum(entry["adapter"]["A"].nbytes + entry["adapter"]["B"].nbytes for entry in pool.entries.values())  # 统计最终活动 Pool 的低秩参数字节数。
slora_active_bytes = base_weight.nbytes + active_adapter_bytes  # 统计基础权重加最终三个驻留 Adapter 的活动内存。
catalog_reduction = 1.0 - slora_catalog_bytes / baseline_resident_bytes  # 计算共享目录相对完整物化的内存降幅。
print("请求       Pool动作  evicted                                        max|baseline-slora|")  # 输出逐请求对照表头。
for row in result_rows:  # 逐条展示数值一致性和缓存行为。
    print(f"{row['id']:<10} {row['action']:<8} {str(row['evicted']):<46} {row['difference']:>20.12f}")  # 输出当前请求的真实差值。
print(f"结果解读：完整物化={baseline_resident_bytes} bytes，S-LoRA目录={slora_catalog_bytes} bytes，活动Pool={slora_active_bytes} bytes，目录降幅={catalog_reduction:.1%}。")  # 量化基础权重共享带来的内存收益。

请求       Pool动作  evicted                                        max|baseline-slora|
slora-01   load     None                                                 0.000000000000
slora-02   load     None                                                 0.000000000000
slora-03   hit      None                                                 0.000000000000
slora-04   load     None                                                 0.000000000000
slora-05   load     ('tenant-a', 'support-tone', 2)                      0.000000000000
slora-06   load     ('tenant-a', 'support-tone', 1)                      0.000000000000
slora-07   load     ('tenant-a', 'legal-brief', 2)                       0.000000000000
slora-08   load     ('tenant-c', 'retail-copy', 3)                       0.000000000000
结果解读：完整物化=9216 bytes，S-LoRA目录=3776 bytes，活动Pool=2656 bytes，目录降幅=59.0%。


## 5. 失败案例与修正：同名跨租户和版本覆盖

只以 `adapter_id` 建表时，tenant-b 的 `support-tone` 覆盖 tenant-a；只以 `(tenant, adapter_id)` 建表时，tenant-a v2 又覆盖 v1。两种错误都会返回数值合理却语义错误的输出。

In [5]:
naive_name_registry = {adapter["adapter_id"]: adapter for adapter in adapters}  # 错误地忽略租户和版本建立注册表。
naive_unversioned_registry = {(adapter["tenant"], adapter["adapter_id"]): adapter for adapter in adapters}  # 错误地忽略版本建立租户内注册表。
collision_request = requests[0]  # 选择 tenant-a support-tone v1 请求复现串权重。
wrong_tenant_adapter = naive_name_registry[collision_request["key"][1]]  # 按名称误取最后写入的 tenant-b Adapter。
wrong_version_adapter = naive_unversioned_registry[(collision_request["key"][0], collision_request["key"][1])]  # 按租户和名称误取 tenant-a v2。
wrong_tenant_output = collision_request["input"] @ (base_weight + wrong_tenant_adapter["A"] @ wrong_tenant_adapter["B"])  # 用跨租户错误参数计算输出。
wrong_version_output = collision_request["input"] @ (base_weight + wrong_version_adapter["A"] @ wrong_version_adapter["B"])  # 用错误版本参数计算输出。
correct_output = baseline_outputs[collision_request["id"]]  # 读取完整键得到的正确 v1 输出。
tenant_collision_difference = float(np.max(np.abs(wrong_tenant_output - correct_output)))  # 量化跨租户串权重影响。
version_collision_difference = float(np.max(np.abs(wrong_version_output - correct_output)))  # 量化旧版本请求被覆盖的影响。
print(f"错误行为1：请求={collision_request['key']}，仅名称命中={wrong_tenant_adapter['key']}，输出最大差={tenant_collision_difference:.6f}")  # 展示跨租户 Adapter 泄漏。
print(f"错误行为2：请求={collision_request['key']}，忽略版本命中={wrong_version_adapter['key']}，输出最大差={version_collision_difference:.6f}")  # 展示版本覆盖。
print(f"修正行为：完整键命中={adapter_registry[collision_request['key']]['key']}，与物化路径差={result_rows[0]['difference']:.12f}")  # 展示租户和版本作用域修正。

错误行为1：请求=('tenant-a', 'support-tone', 1)，仅名称命中=('tenant-b', 'support-tone', 1)，输出最大差=0.042500
错误行为2：请求=('tenant-a', 'support-tone', 1)，忽略版本命中=('tenant-a', 'support-tone', 2)，输出最大差=0.068018
修正行为：完整键命中=('tenant-a', 'support-tone', 1)，与物化路径差=0.000000000000


## 6. 生产边界

本例只是一层 NumPy 线性变换，没有真实 GPU kernel。生产 S-LoRA 需要预分配连续显存页、融合 base/delta kernel、不同 rank batching、版本 pin 和原子发布、签名鉴权、租户配额、热度预取、公平调度、OOM 降级与 p95/p99 延迟监控。

In [6]:
hit_count = sum(event["action"] == "hit" for event in pool.events)  # 统计八条请求中的 Adapter Pool 命中数。
eviction_count = sum(event["evicted"] is not None for event in pool.events)  # 统计容量为三时发生的 LRU 淘汰次数。
diagnostics = {"requests": len(requests), "adapters": len(adapters), "pool_capacity": pool.capacity, "hits": hit_count, "evictions": eviction_count, "catalog_memory_reduction": catalog_reduction, "maximum_output_difference": max(row["difference"] for row in result_rows), "scoped_key_fields": ("tenant", "adapter_id", "version")}  # 汇总多 Adapter Serving 质量和容量指标。
print("生产监控快照：", diagnostics)  # 输出上线时应持续观察的 Serving 指标。

生产监控快照： {'requests': 8, 'adapters': 6, 'pool_capacity': 3, 'hits': 1, 'evictions': 4, 'catalog_memory_reduction': 0.5902777777777778, 'maximum_output_difference': 2.220446049250313e-16, 'scoped_key_fields': ('tenant', 'adapter_id', 'version')}


## 7. 最小回归测试

断言验证案例规模、两条前向等价、内存收益、Pool 行为以及租户和版本隔离。

In [7]:
assert len(adapters) >= 5 and len(requests) >= 5  # 保证案例覆盖多个 Adapter 和请求。
assert max(row["difference"] for row in result_rows) < 1.0e-10  # 保证共享 base 加低秩增量与完整物化数值等价。
assert slora_catalog_bytes < baseline_resident_bytes and slora_active_bytes < slora_catalog_bytes  # 保证目录共享和容量池都体现内存收益。
assert hit_count >= 1 and eviction_count >= 1  # 保证真实复现 Pool 命中与 LRU 换出。
assert wrong_tenant_adapter["tenant"] != collision_request["key"][0] and tenant_collision_difference > 1.0e-6  # 保证仅名称索引会真实串到另一租户并改变输出。
assert wrong_version_adapter["version"] != collision_request["key"][2] and version_collision_difference > 1.0e-6  # 保证忽略版本会真实覆盖 v1 输出。